# Exploring the GIS test data (permits, service requests, road segments)

There is no data dictionary, so before setting up the ArcGIS Pro project I checked the three layers for the decisions the geoprocessing depends on:

1. **Load:** what's in each layer, and are the key columns complete?
2. **Permits (layer 23):** which columns mean renewal, issued, excavation and paving, and how many permits pass?
3. **Service requests (Cityworks):** does the provided SQL match the brief, and how many requests pass?
4. **Road segments (layer 162):** which road types count as streets, service lanes and ramps?
5. **Snapping:** how far are the permit points from the road?

Each section ends with the decision it led to.

*This notebook uses the Python test's environment (`python_test/.venv`, kernel "CS12 Python Test (3.11)"), since it needs the same two packages, `requests` and `pandas`. No `arcpy`, so it runs outside ArcGIS Pro.*

In [1]:
import pandas as pd
import requests

pd.set_option("display.width", 160)

PERMITS_23 = "https://maps2.dcgis.dc.gov/dcgis/rest/services/DCGIS_APPS/DDOT_TOPS/MapServer/23"
REQUESTS_CW = "https://maps2.dcgis.dc.gov/dcgis/rest/services/DDOT/Cityworks/MapServer/1"
SUBBLOCKS_162 = "https://maps2.dcgis.dc.gov/dcgis/rest/services/DCGIS_DATA/Transportation_WebMercator/MapServer/162"

## 1. Load the data

Three layers this time, so the download code is a small function. The services hand out at most 2,000 rows per request, so it asks page by page.

- **Permits:** the whole layer (about 55,000 rows), so nothing is hidden before I decide how to filter.
- **Service requests:** Cityworks holds about 1.5 million requests going back years. The brief only wants requests created on or after 1 October 2025, so only those are downloaded (about 75,000). Every other filter is decided from the data.
- **Road segments:** the whole layer (about 35,000), without the line shapes for now.

In [2]:
def download(layer_url, where="1=1"):
    # The layer's own description: column names, types and code lists
    layer_info = requests.get(layer_url, params={"f": "json"}, timeout=60).json()

    # Ask for the rows page by page until the service says there are no more
    rows, offset = [], 0
    while True:
        params = {
            "where": where,
            "outFields": "*",
            "returnGeometry": "false",    # the points have their coordinates as columns
            "orderByFields": "OBJECTID",  # same order every time, so pages don't overlap
            "resultOffset": offset,
            "resultRecordCount": 2000,
            "f": "json",
        }
        page = requests.get(f"{layer_url}/query", params=params, timeout=120).json()
        if "error" in page:               # a bad query still comes back as "OK"
            raise RuntimeError(page["error"])
        rows += [feature["attributes"] for feature in page["features"]]
        if not page.get("exceededTransferLimit"):
            break
        offset += len(page["features"])
    table = pd.DataFrame(rows)

    # Dates arrive as milliseconds since 1970 in UTC, so convert them to DC time.
    # Coded columns get an extra "_label" column with the readable value.
    for field in layer_info["fields"]:
        name = field["name"]
        if name not in table:
            continue
        if field["type"] == "esriFieldTypeDate":
            table[name] = pd.to_datetime(table[name], unit="ms", utc=True).dt.tz_convert("America/New_York")
        domain = field.get("domain") or {}
        if domain.get("type") == "codedValue":
            labels = {str(c["code"]): c["name"] for c in domain["codedValues"]}
            table[name + "_label"] = table[name].astype("string").str.removesuffix(".0").map(labels)
    return table

In [3]:
permits = download(PERMITS_23)
service_requests = download(REQUESTS_CW, where="INITIATEDDATE >= timestamp '2025-10-01 00:00:00'")
subblocks = download(SUBBLOCKS_162)

for name, table in [("Permits (layer 23)", permits), ("Service requests since 1 Oct 2025", service_requests),
                    ("Road segments (layer 162)", subblocks)]:
    print(f"{name}: {len(table):,} rows, {table.shape[1]} columns")

Permits (layer 23): 54,625 rows, 102 columns
Service requests since 1 Oct 2025: 74,944 rows, 57 columns
Road segments (layer 162): 34,964 rows, 113 columns


In [4]:
# How complete are the columns this task depends on? (share of rows filled)
permit_columns = ["TrackingNumber", "PermitNumber", "StatusDescription", "IssueDate", "ExpirationDate",
                  "Renewal", "RenewalDescription", "IsExcavation", "IsPaving", "PermitTypeDescription",
                  "WorkLocationFullAddress", "XCoord", "YCoord", "OnSegX", "OnSegY", "StreetSegID", "RoadwaySegID"]
request_columns = ["REQUESTID", "DESCRIPTION", "STATUS", "INITIATEDDATE", "INSPECTIONDATE", "INSPECTIONCOMPLETE",
                   "CLOSEDDATE", "ADDRESS", "XCOORD", "YCOORD", "ONSEGX", "ONSEGY"]
subblock_columns = ["SUBBLOCKKEY", "ROUTENAME", "ROADTYPE"]

pd.concat([
    permits[permit_columns].notna().mean().round(3).rename("share filled").to_frame().assign(layer="permits"),
    service_requests[request_columns].notna().mean().round(3).rename("share filled").to_frame().assign(layer="service requests"),
    subblocks[subblock_columns].notna().mean().round(3).rename("share filled").to_frame().assign(layer="road segments"),
])

,share filled,layer
TrackingNumber,1.000,permits
PermitNumber,0.659,permits
StatusDescription,1.000,permits
IssueDate,0.659,permits
ExpirationDate,1.000,permits
Renewal,1.000,permits
RenewalDescription,1.000,permits
IsExcavation,1.000,permits
IsPaving,1.000,permits
PermitTypeDescription,0.866,permits


**What this shows**

- All three layers download cleanly: 54,625 permits, 74,944 service requests since 1 October 2025, and 34,964 road segments.
- The columns the filters need are always filled: `ExpirationDate`, `Renewal`, `IsExcavation` and `IsPaving` for permits; `DESCRIPTION`, `STATUS`, `INITIATEDDATE` and `INSPECTIONCOMPLETE` for requests; `ROADTYPE` for road segments.
- `IssueDate` and `PermitNumber` are filled on 66% of permits. As in layer 24, they probably only exist once a permit is issued, which section 2 checks.
- Permits can't be joined to road segments by an ID: `RoadwaySegID` is always empty and `StreetSegID` is filled on only 59%, and neither is the road segment key (`SUBBLOCKKEY`).

**Decision:** permits and requests are linked to road segments **by location** (snapping and a spatial join in ArcGIS Pro), not by a shared ID.